In [ ]:
from scapy.all import *
import time

dns_counter = {}

THRESHOLD = 5
WINDOW = 10

def alert_dnssnooping(pkt):

    if pkt.haslayer(DNSQR):

        src_ip = pkt[IP].src
        query = pkt[DNSQR].qname.decode()

        now = time.time()

        if src_ip not in dns_counter:
            dns_counter[src_ip] = []

        dns_counter[src_ip].append(now)

        # eliminar consultas antiguas
        dns_counter[src_ip] = [
            t for t in dns_counter[src_ip]
            if now - t < WINDOW
        ]

        print(f"[DNS] {src_ip} -> {query}")

        # deteccion por volumen
        if len(dns_counter[src_ip]) > THRESHOLD:
            print("\n[!!!] Posible DNS Snooping detectado")
            print(f"IP sospechosa: {src_ip}")
            print(f"Consultas DNS en {WINDOW} segundos: {len(dns_counter[src_ip])}")
            print("------------------------------------")

print("[*] Monitorizando consultas DNS...")

sniff(filter="udp port 53", prn=alert_dnssnooping, store=0)
